In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.preprocessing import MinMaxScaler
from fbm import FBM

from utils.config import load_config


In [ ]:
config = load_config()


In [ ]:
def get_pandas_dataframe_size_in_gb(df):
    return df.memory_usage(deep=True).sum() / (1024**3)


def generate_fbm_sample(n: int, hurst: float):
    return FBM(n=n, hurst=hurst, method="hosking").fbm()[
        1:
    ]  # [1:] to exclude initial 0


def gen_data(samples_num: int, sample_size: int) -> pd.DataFrame:
    """Generate fBm data with random subsampling of given size from longer trajectory."""
    data = []
    hurst_params = np.random.uniform(0.001, 0.999, size=samples_num).round(3)
    start_indices = np.random.randint(
        0, config.sample_to_subsample_size - sample_size, size=samples_num
    )

    scaler = MinMaxScaler(feature_range=(0, 1))

    for start_index, hurst_param in tqdm(
        zip(start_indices, hurst_params),
        desc=f"Generating subsamples of size {sample_size}",
    ):
        full_sample = generate_fbm_sample(config.sample_to_subsample_size, hurst_param)
        subsample = full_sample[start_index : start_index + sample_size].reshape(-1, 1)
        normalized = scaler.fit_transform(subsample).flatten()
        data.append(np.append(normalized, hurst_param))

    columns = [f"x{i}" for i in range(sample_size)] + [config.target_column]
    df = (
        pd.DataFrame(data, columns=columns)
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    return df


In [ ]:
os.makedirs(f"{config.data_dir}/{config.train_data_subdir}/", exist_ok=True)

for sample_size in config.sample_sizes:
    print(
        f"Start generating data for sample size {sample_size} x {config.train_samples_num} samples"
    )

    data = gen_data(samples_num=config.train_samples_num, sample_size=sample_size)

    path_to_save = f"./{config.data_dir}/{config.train_data_subdir}/fbm_{sample_size}x{config.train_samples_num}.csv"

    print(f"Saving data to '{path_to_save}'")

    data.to_csv(path_to_save, index=False)

    print(f"Size: {get_pandas_dataframe_size_in_gb(data): .4f}GB")

    print("Complete.")


In [ ]:
os.makedirs(f"{config.data_dir}/{config.test_data_subdir}/", exist_ok=True)

for sample_size in config.sample_sizes:
    print(
        f"Start generating data for sample size {sample_size} x {config.test_samples_num} samples"
    )

    data = gen_data(samples_num=config.test_samples_num, sample_size=sample_size)

    path_to_save = f"./{config.data_dir}/{config.test_data_subdir}/fbm_{sample_size}x{config.test_samples_num}.csv"

    print(f"Saving data to '{path_to_save}'")

    data.to_csv(path_to_save, index=False)

    print(f"Size: {get_pandas_dataframe_size_in_gb(data): .4f}GB")

    print("Complete.")
